# Road Damage Detection and Classification using YOLOv8
### Project-Based Learning (PBL) | India Road Distress Dataset
**Author:** Ravish Patel  
**Objective:** Automate detection of road cracks (D00, D10, D20) and potholes (D40) on Indian roads using Deep Learning and Computer Vision.

## 1. Mount Google Drive
We mount Google Drive to access our `RoadDamagePBL` folder where `raw_data`, `dataset`, and `models` reside.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Change directory to the RoadDamagePBL project folder
project_dir = '/content/drive/MyDrive/RoadDamagePBL'
if os.path.exists(project_dir):
    os.chdir(project_dir)
    print(f"[✓] Current Working Directory: {os.getcwd()}")
else:
    print(f"[!] Folder {project_dir} not found. Please verify Google Drive path.")

## 2. Verify GPU Acceleration
Ensure runtime is set to GPU (Runtime -> Change runtime type -> T4 GPU).

In [ ]:
!nvidia-smi

## 3. Install Dependencies
Install Ultralytics YOLOv8 and associated packages.

In [ ]:
!pip install -q ultralytics pyyaml matplotlib

## 4. Dataset Preparation: Convert VOC XML to YOLO Format
This script parses the Pascal VOC XML files from `raw_data/India/train/annotations/xmls`,
normalizes coordinates into YOLO format `[class_id, x_center, y_center, width, height]`,
splits into 80% train and 20% validation sets, and writes `data.yaml`.

In [ ]:
!python voc_to_yolo.py --base_dir . --output_dir dataset --split 0.8

## 5. Inspect Dataset Configuration (`data.yaml`)

In [ ]:
with open('data.yaml', 'r') as f:
    print(f.read())

## 6. Model Training (YOLOv8)
We train YOLOv8 (using pre-trained COCO backbone `yolov8n.pt` for transfer learning) for 25 epochs.

In [ ]:
!python train.py --data data.yaml --model yolov8n.pt --epochs 25 --batch 16 --imgsz 640 --device 0

## 7. Model Evaluation
Compute Precision, Recall, and Mean Average Precision (mAP@0.50 and mAP@0.50:0.95).

In [ ]:
!python evaluate.py --weights models/best.pt --data data.yaml --device 0

## 8. Visualize Training Metrics & Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

metrics_images = ['results/screenshots/results.png', 'results/screenshots/confusion_matrix.png']
for p in metrics_images:
    if os.path.exists(p):
        plt.figure(figsize=(10, 6))
        plt.imshow(mpimg.imread(p))
        plt.axis('off')
        plt.title(os.path.basename(p))
        plt.show()

## 9. Run Inference on Test / Demo Images

In [ ]:
!python predict.py --source demo_samples --weights models/best.pt --conf 0.25

## 10. Visualize Sample Detections

In [ ]:
import glob
preds = glob.glob('results/predictions/*.jpg')[:4]
for img_file in preds:
    plt.figure(figsize=(8, 8))
    plt.imshow(mpimg.imread(img_file))
    plt.axis('off')
    plt.title(f"Detected: {os.path.basename(img_file)}")
    plt.show()